<div style="background: linear-gradient(120deg,#052e16,#14532d); border-radius:16px; padding:36px 40px; color:#e2e8f0;">
<h1 style="margin:0; font-size:2.1em; color:#ffffff;">🕸️ Graph Databases</h1>
<h3 style="margin:6px 0 0 0; font-weight:400; color:#86efac;">Session 3 of 4 — your knowledge has a shape</h3>
<p style="margin-top:18px; color:#cbd5e1;">NordWind Energy Workshop Series · Retrieval, Vectors & Graphs</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noctetemp/nordwind-workshop/blob/main/session3_graphs_en.ipynb)

## 🗺️ Today's journey

Two sessions of evidence, one verdict: *points in space have no edges.* Your own field report said it best — "performing accurate relationship traversal or exhaustive retrieval with vector search alone is unstable." Today, edges become **first-class citizens**.

* **🔗 The relational pain** — why "friends of friends of friends" makes SQL cry: the JOIN explosion, shown side by side with its one-line graph equivalent.
* **✨ The living graph** — the moment you've been building toward: all of NordWind — every team, engineer, service and incident — as a draggable, physics-driven network. You will *recognize* things in it before anyone explains them.
* **🔌 Neo4j** — connect to your own cloud instance (Aura), load the world in seconds.
* **🎨 Cypher** — the query language that is literally ASCII art: `(:Engineer)-[:RESPONDED_TO]->(:Incident)`. You draw the pattern; the database finds every place it occurs.
* **🪜 The query ladder** — seven queries, escalating from "look up a service" to things that would be genuinely painful in SQL. Variable-length paths, aggregation over structure, cross-team analysis.
* **⚔️ THE HARD QUESTION** — the one that broke naive RAG in Session 1 and survived RAG v2 in Session 2 — answered **completely, in four lines, with the reasoning attached**.
* **🛰️ The Graph Navigator** — the same database, as a starship's tactical display: click a node and watch Neo4j answer in 3D. Then the **RAG autopsy**: we put Session 1's wrong answers on the map and *see* why vectors failed.
* **🧠 When to reach for a graph** — and when not to. Modeling mindset: nouns become nodes, *verbs become relationships*.

> ⚠️ **Pre-session requirement:** a free Neo4j Aura instance (takes 3 minutes — see `AURA_SETUP.md` in the repo). No Aura? The visual sections work without it, and you can pair up for the query cells.

## 0 · Setup

In [ ]:
%pip -q install neo4j pyvis
print("✅ ready")

In [ ]:
import json, urllib.request

BASE = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/"
def fetch(path):
    return json.loads(urllib.request.urlopen(BASE + path).read())

teams     = fetch("entities/teams.json")
engineers = fetch("entities/engineers.json")
services  = fetch("entities/services.json")
incidents = fetch("entities/incidents.json")
rels      = fetch("relationships.json")
print(f"🌍 NordWind: {len(teams)} teams · {len(engineers)} engineers · "
      f"{len(services)} services · {len(incidents)} incidents · {len(rels)} relationships")

---
## 1 · 🔗 Why not just... SQL?

Fair question — relational databases *can* store relationships. A `service_dependencies(from_id, to_id)` table is a graph, technically. The pain starts when you **traverse**.

*"Which services could an outage of `payment-gateway` ultimately impact?"* — that's dependents, of dependents, of dependents. In SQL:

```sql
-- one hop
SELECT s2.name FROM services s1
JOIN service_dependencies d1 ON d1.to_id = s1.id
JOIN services s2 ON s2.id = d1.from_id
WHERE s1.name = 'payment-gateway';

-- two hops: JOIN the join. three hops: JOIN the join of the join.
-- unknown depth: recursive CTE, ~15 lines, and each JOIN is an index
-- lookup whose cost grows with table size.
```

The same question in **Cypher**, at *any* depth:

```cypher
MATCH (down:Service)-[:DEPENDS_ON*1..]->(:Service {name: 'payment-gateway'})
RETURN DISTINCT down.name
```

Two deeper reasons this isn't just syntax sugar:

1. **Index-free adjacency.** In a native graph store, each node holds direct pointers to its neighbors. Hopping an edge is pointer-chasing — constant cost per hop, *independent of how big the database is*. A JOIN, by contrast, is an index lookup that gets more expensive as tables grow.
2. **The relationship carries meaning.** `DEPENDS_ON`, `OWNS`, `RESPONDED_TO` are typed, directed, and can hold properties — they're data, not plumbing.

But before any more theory — you should *see* what you've been working with for two sessions.

---
## 2 · ✨ The living graph

Same 153 relationships you computed set-operations over in Session 1. This time, drawn — with physics.

🟡 hexagons = **teams** · 🔵 dots = **engineers** · 🟩 boxes = **services** · 🔴 diamonds = **incidents**

**Drag things. Zoom. Hover.** Take two full minutes before scrolling — find yourself a story in there.

In [ ]:
from pyvis.network import Network
from IPython.display import HTML, display

COLORS = {"Team": "#f59e0b", "Engineer": "#60a5fa", "Service": "#34d399", "Incident": "#f87171"}

def build_net():
    net = Network(height="720px", width="100%", bgcolor="#0f172a",
                  font_color="#e2e8f0", directed=True, cdn_resources="remote")
    net.barnes_hut(gravity=-9000, central_gravity=0.25,
                   spring_length=140, spring_strength=0.02, damping=0.35)
    for t in teams:
        net.add_node(t["name"], label=t["name"], color=COLORS["Team"], shape="hexagon",
                     size=34, title=f"Team — {t['focus']}")
    for e in engineers:
        net.add_node(e["name"], label=e["name"].split()[0], color=COLORS["Engineer"],
                     size=16, title=f"{e['name']} — {e['role']}, {e['team_name']}")
    for s in services:
        net.add_node(s["name"], label=s["name"], color=COLORS["Service"], shape="box",
                     size=24, title=f"{s['description']} (owner: {s['owner_team']})")
    for i in incidents:
        net.add_node(i["id"], label=i["id"], color=COLORS["Incident"], shape="diamond",
                     size=20, title=f"{i['severity']} — {i['title']}")
    return net

EDGE_STYLE = {"MEMBER_OF": {"color": "#64748b", "width": 1},
              "OWNS": {"color": "#f59e0b", "width": 2},
              "DEPENDS_ON": {"color": "#34d399", "width": 2, "dashes": True},
              "AFFECTED": {"color": "#f87171", "width": 2},
              "RESPONDED_TO": {"color": "#93c5fd", "width": 1}}

net = build_net()
for r in rels:
    net.add_edge(r["from"], r["to"], title=r["type"], **EDGE_STYLE[r["type"]])

display(HTML(net.generate_html(notebook=False)))

🎤 **What did you find?** *(Facilitator: before scrolling, ask the room "which service would you fix first?" — make them commit to a guess. In ten minutes a query will grade it.)*

Some things people usually spot: the billing-engine sits in a dense red thicket (it *is* NordWind's most incident-prone service — you'll prove it with a query in ten minutes); teams form natural neighborhoods because ownership and membership pull them together; and a few engineers sit *between* clusters — the cross-team responders.

Here's the point worth pausing on: **this structure was in your documents all along.** Sessions 1–2 read these facts as flat text. Nothing was added tonight except *shape*.

Now let's make this queryable.

---
## 3 · 🔌 Connect to your Neo4j

Fill in the credentials from your Aura setup (the `.txt` file you downloaded when creating the instance — see `AURA_SETUP.md`). The URI looks like `neo4j+s://xxxxxxxx.databases.neo4j.io`.

In [ ]:
NEO4J_URI      = ""  # @param {type:"string"}
NEO4J_PASSWORD = ""  # @param {type:"string"}
NEO4J_USER     = "neo4j"

from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

def cypher(q, **params):
    """Run a Cypher query, return a list of dicts."""
    with driver.session() as s:
        return [r.data() for r in s.run(q, **params)]

print("🔌 connected:", cypher("RETURN 'hello from ' + 'Neo4j' AS msg")[0]["msg"])

Loading the world: `UNWIND` turns a Python list into rows, `MERGE` creates each node if it doesn't already exist (safe to re-run). Nodes first, then relationships — a relationship needs both of its endpoints to exist:

In [ ]:
cypher("MATCH (n) DETACH DELETE n")   # clean slate — safe on your empty workshop instance

cypher("UNWIND $rows AS r MERGE (t:Team {name: r.name}) SET t.focus = r.focus", rows=teams)
cypher("UNWIND $rows AS r MERGE (e:Engineer {name: r.name}) SET e.role = r.role", rows=engineers)
cypher("UNWIND $rows AS r MERGE (s:Service {name: r.name}) "
       "SET s.description = r.description, s.language = r.language", rows=services)
cypher("UNWIND $rows AS r MERGE (i:Incident {id: r.id}) "
       "SET i.title = r.title, i.severity = r.severity, i.date = date(r.date)", rows=incidents)

by = lambda t: [r for r in rels if r["type"] == t]
cypher("UNWIND $rows AS r MATCH (e:Engineer {name: r.from}), (t:Team {name: r.to}) "
       "MERGE (e)-[:MEMBER_OF]->(t)", rows=by("MEMBER_OF"))
cypher("UNWIND $rows AS r MATCH (t:Team {name: r.from}), (s:Service {name: r.to}) "
       "MERGE (t)-[:OWNS]->(s)", rows=by("OWNS"))
cypher("UNWIND $rows AS r MATCH (a:Service {name: r.from}), (b:Service {name: r.to}) "
       "MERGE (a)-[:DEPENDS_ON]->(b)", rows=by("DEPENDS_ON"))
cypher("UNWIND $rows AS r MATCH (i:Incident {id: r.from}), (s:Service {name: r.to}) "
       "MERGE (i)-[:AFFECTED]->(s)", rows=by("AFFECTED"))
cypher("UNWIND $rows AS r MATCH (e:Engineer {name: r.from}), (i:Incident {id: r.to}) "
       "MERGE (e)-[:RESPONDED_TO]->(i)", rows=by("RESPONDED_TO"))

print(cypher("MATCH (n) RETURN count(n) AS nodes")[0],
      cypher("MATCH ()-[r]->() RETURN count(r) AS rels")[0])

**73 nodes, 153 relationships.** The entire company, loaded in under two seconds. 💡 *Also open your Aura console → Query tab and run `MATCH (n) RETURN n` there — Aura's built-in visualization gives you the draggable graph connected live to the database.*

---
## 4 · 🎨 Cypher — queries you can draw

Cypher's core idea: **you draw the shape you're looking for, and the database finds every occurrence of it.**

```
(:Engineer)-[:RESPONDED_TO]->(:Incident)-[:AFFECTED]->(:Service)
   node ──────── edge ─────────► node ────── edge ──────► node
```

`()` are nodes, `-[]->` are relationships, `{...}` pin properties. That's 80% of the language. Let's climb the ladder — each query one step less possible for vector search:

In [ ]:
# Rung 1 — lookup (any database can do this)
cypher("MATCH (s:Service {name: 'payment-gateway'}) RETURN s.description AS description")

In [ ]:
# Rung 2 — one hop: who owns billing-engine?
cypher("MATCH (t:Team)-[:OWNS]->(s:Service {name: 'billing-engine'}) RETURN t.name AS owner")

In [ ]:
# Rung 3 — one hop + aggregation: the full Payments & Billing roster
cypher("""MATCH (e:Engineer)-[:MEMBER_OF]->(t:Team {name: 'Payments & Billing'})
          RETURN collect(e.name) AS roster""")

In [ ]:
# Rung 4 — blast radius: everything INC-2111 touched
cypher("""MATCH (i:Incident {id: 'INC-2111'})-[:AFFECTED]->(s:Service)
          RETURN i.title AS incident, collect(s.name) AS affected_services""")

In [ ]:
# Rung 5 — VARIABLE-LENGTH paths: everything customer-portal transitively depends on.
# *1..3 means "follow 1 to 3 DEPENDS_ON edges" — this is the recursive-CTE killer.
cypher("""MATCH (:Service {name: 'customer-portal'})-[:DEPENDS_ON*1..3]->(up:Service)
          RETURN collect(DISTINCT up.name) AS upstream_dependencies""")

Read Rung 5's result carefully: customer-portal transitively depends on **payment-gateway** and even **meter-reader** — three hops away, through billing-engine. No document states this anywhere. It's *derived from structure*. Your team's field report asked for exactly this: exhaustive relationship exploration that doesn't miss.

In [ ]:
# Rung 6 — aggregation over structure: NordWind's most incident-prone services
cypher("""MATCH (i:Incident)-[:AFFECTED]->(s:Service)
          RETURN s.name AS service, count(i) AS incident_count
          ORDER BY incident_count DESC LIMIT 5""")

In [ ]:
# Rung 7 — structural analysis: incidents that required responders from MULTIPLE teams
# (a proxy for organizational complexity — pure graph thinking)
cypher("""MATCH (e:Engineer)-[:MEMBER_OF]->(t:Team), (e)-[:RESPONDED_TO]->(i:Incident)
          WITH i, count(DISTINCT t) AS teams_involved
          WHERE teams_involved > 1
          RETURN count(i) AS cross_team_incidents""")

13 of 20 incidents needed more than one team. Try asking a vector index that. 😄

Notice what happened to *counting and completeness* — the two things your field report flagged as unreliable. `count`, `collect`, `DISTINCT` over a graph are **exact**: not "the most relevant results", but *the* results. All of them, provably.

---
## 5 · ⚔️ The hard question — the payoff

Three sessions ago, this question broke naive RAG. Last session, it survived hybrid search *and* reranking. It needs: dependency knowledge → incident linkage → exhaustive responder aggregation. A join across three relationship types.

In Cypher, you simply... draw it:

In [ ]:
cypher("""
MATCH (dep:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'})
MATCH (i:Incident)-[:AFFECTED]->(dep)
MATCH (e:Engineer)-[:RESPONDED_TO]->(i)
WITH e, i ORDER BY e.name
RETURN collect(DISTINCT e.name) AS engineers,
       count(DISTINCT i)        AS qualifying_incidents
""")

**All 10 engineers. All 6 incidents. Four lines. Milliseconds. Deterministic** — run it a thousand times, get the same answer a thousand times, something no similarity ranking can promise.

And unlike RAG's confident partial answers, this one can *show its work*. Ask the graph **why** an engineer qualifies:

In [ ]:
cypher("""
MATCH p = (e:Engineer {name: 'Winry Rockbell'})-[:RESPONDED_TO]->(:Incident)
          -[:AFFECTED]->(:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'})
RETURN [n IN nodes(p) | coalesce(n.name, n.id)] AS evidence_path
""")

`Winry Rockbell → INC-2105 → billing-engine → payment-gateway` — the answer **with its reasoning attached**. In regulated or safety-relevant contexts, this explainability is not a nice-to-have; it's the requirement.

Now let's *see* the answer — the same living graph, but with every evidence path lit up:

In [ ]:
# recompute the answer set in Python (same 3 hops, set-style) to drive the highlight
dependents  = {r["from"] for r in rels if r["type"] == "DEPENDS_ON" and r["to"] == "payment-gateway"}
qual_incs   = {r["from"] for r in rels if r["type"] == "AFFECTED" and r["to"] in dependents}
answer_engs = {r["from"] for r in rels if r["type"] == "RESPONDED_TO" and r["to"] in qual_incs}
lit = answer_engs | qual_incs | dependents | {"payment-gateway"}

net2 = build_net()
for n in net2.nodes:                       # dim everything...
    if n["id"] not in lit:
        n["color"] = "#334155"; n["opacity"] = 0.25
    else:                                  # ...then relight the answer
        n["size"] = n.get("size", 16) + 10
        n["borderWidth"] = 3

def on_path(r):
    return ((r["type"] == "RESPONDED_TO" and r["from"] in answer_engs and r["to"] in qual_incs) or
            (r["type"] == "AFFECTED"     and r["from"] in qual_incs   and r["to"] in dependents) or
            (r["type"] == "DEPENDS_ON"   and r["from"] in dependents  and r["to"] == "payment-gateway"))

for r in rels:
    if on_path(r):
        net2.add_edge(r["from"], r["to"], color="#fbbf24", width=4, title=r["type"])
    else:
        net2.add_edge(r["from"], r["to"], color="#1e293b", width=1)

display(HTML(net2.generate_html(notebook=False)))

**That golden web is the answer.** Ten engineers, six incidents, one dependency edge — the join, drawn in light through the noise of everything that didn't matter. This picture is what "retrieval by structure" *means*, and it's the picture to remember whenever someone asks you "why graphs?".

---
## 5b · 🛰️ The Graph Navigator — watch the database answer

Everything so far returned tables. Now the same database, as a ship's tactical display:

**👉 [Open the Graph Navigator](https://raw.githack.com/noctetemp/nordwind-workshop/main/nordwind_3d.html)** — press **🔌 UPLINK** and paste the *same* URI and password you used in section 3. The page holds **no data**: every sphere you're about to see is a Cypher result that arrived milliseconds earlier (the log at the bottom shows the record count and timing for every query).

One green node appears alone in the void — `payment-gateway`. Now **navigate**:

1. **Click it.** The Navigator runs `MATCH (n)-[r]-(m)` for that node; its neighbors materialize. billing-engine is among them.
2. **Click billing-engine.** customer-portal appears.
3. Pause here. *No document anywhere says customer-portal depends on payment-gateway.* You derived it by walking two edges — **click = hop**. That's all "traversal" ever means, and it's what Rung 5 did in one line.
4. Take a request from the room — *"show Levi Ackerman"* — and type it in the console:
   `MATCH (e:Engineer {name:'Levi Ackerman'})-[r]-(m) RETURN e,r,m`
5. Press **⚡ TRACE IMPACT**. It runs the hard question as a real path query; the golden web from the previous cell appears in three dimensions, pulled live from your database.

**The guess you made earlier** — which service to fix first? Press **⬇ LOAD ALL**, let it orbit, and look at the red thicket. Then look at Rung 6's table. Intuition and query agree *here*. Keep that in mind for the next cell, where they don't.

### 🔬 The RAG autopsy

In Session 1, naive RAG answered the hard question with **INC-2117, INC-2107 and INC-2113** — confidently, and wrong. We said "it retrieved incidents that *sounded like* payments." Now we can do better than a metaphor: we can put the wrong answers *on the map*.

The question asks for incidents on services that **depend on** payment-gateway. Let's ask the graph whether RAG's three picks qualify:

In [ ]:
cypher("""
MATCH (i:Incident) WHERE i.id IN ['INC-2117', 'INC-2107', 'INC-2113']
RETURN i.id  AS incident,
       [(i)-[:AFFECTED]->(x) | x.name]  AS affected_services,
       EXISTS { (i)-[:AFFECTED]->(:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'}) }
                                          AS qualifies
ORDER BY incident
""")

Three rows, three `false` — and *look at why*:

- **INC-2117 and INC-2113 hit `payment-gateway` itself.** They're about the question's *subject*, not its *answer*. The question asks about services one hop *upstream* — the vectors couldn't tell "about X" from "connected to X". No embedding encodes the direction of an edge.
- **INC-2107 hit `api-gateway`.** A different gateway entirely. The word "gateway" was enough to pull it in — lexical resemblance masquerading as relevance.

Now make the room *see* it. In the Navigator, with TRACE still lit, paste this into the console:

```cypher
MATCH (i:Incident) WHERE i.id IN ['INC-2117','INC-2107','INC-2113'] MATCH (i)-[r]-(m) RETURN i, r, m
```

Two of the three light up *touching* payment-gateway — adjacent to the hub, but **not on any golden path**. The third floats off by a different gateway. RAG found text that *resembled the question*; the graph found nodes *connected to the answer*. That one picture is the whole argument of Sessions 1–3.

> 🧭 **A confession about pretty pictures.** This is stunning at 73 nodes. At 70,000 it is a hairball and your eyes are useless — you'd never spot billing-engine's thicket or the three impostor incidents. That is not a weakness of the tool; it's the *reason query languages exist*. Looking answers **shape** questions (where's the hub? what's a cluster?). Only queries answer **list** questions (which ten engineers, exactly?). Use the picture to build intuition, and Cypher to be right.

> 🔭 **Notice what you just did with the mouse.** You started from a *named* node, then expanded outward a few hops, collecting what was connected. Hold onto that gesture. Next session, a vector search will choose the starting node for you ("where to begin") and the graph will do the expansion ("what's connected") — and that is the entire GraphRAG algorithm. You've already performed it by hand.

---
## 6 · 🧠 When to reach for a graph — and when not to

**Modeling mindset:** listen to the domain language. **Nouns become nodes** (Engineer, Service, Incident). **Verbs become relationships** (owns, depends on, responded to). If your stakeholders naturally say "X *is connected to* Y through Z", you're being told the schema.

| Reach for a graph when... | Stay with tables / vectors when... |
|---|---|
| Questions traverse: "through", "connected to", "impact of", "who worked with whom" | Questions filter & aggregate flat records: "revenue by region" |
| Depth is unknown or variable (`*1..`) | Joins are shallow and fixed (1–2 known hops) |
| Completeness and explainability are required | "Best matches" are good enough |
| Relationships are the data (org charts, dependencies, fraud rings, lineage) | Relationships are incidental foreign keys |
| — | The content is unstructured prose → that's what **vectors** are for |

That last row matters most: **graphs and vectors are not competitors.** The graph knows *who is connected to what*; the vectors know *what the documents say*. NordWind's graph can name INC-2105's responders but hasn't read the postmortem's prose; the vector index has read everything but sees no structure. You want both.

## 🏁 What you learned today

- Traversal-shaped questions make SQL scream and Cypher shrug — index-free adjacency is why
- Cypher = draw the pattern, get every occurrence: `MATCH`, properties, `*1..n` paths, exact aggregation
- The hard question fell in 4 lines, **with evidence paths** — deterministic, complete, explainable
- Nouns → nodes, verbs → relationships; graphs for structure, vectors for prose
- Pictures show **shape**, queries answer **lists** — RAG's wrong answers touched the hub on the map but sat on no path

## 📝 Before next session
The practice notebook has Cypher katas on the NordWind graph — including one question that *neither* the graph *nor* the vectors can answer alone. Find it, and you've understood Session 4 before it starts.

---

<div style="background: linear-gradient(120deg,#450a0a,#7f1d1d); border-radius:16px; padding:30px 36px; color:#e2e8f0;">
<h2 style="margin:0; color:#ffffff;">⏭️ To be continued...</h2>
<p style="font-size:1.05em; margin-top:14px;">Tonight's magic ran on a confession: our graph was loaded from a clean <code>relationships.json</code> that came <i>with</i> the fictional world. Your real company has no such file. Your dependencies live in ADR prose. Your responder lists live in postmortems. Your org chart lives in people's heads.</p>
<p style="font-size:1.05em;">Next session — the finale: an LLM reads NordWind's raw documents and <b>builds this graph itself</b>. Then we fuse everything: vectors find <i>where to start</i>, the graph finds <i>what's connected</i>, and the question that started this workshop gets answered end-to-end from nothing but prose.</p>
<h3 style="color:#fca5a5; margin-bottom:0;">Session 4: GraphRAG — <i>full circle</i> 🔄</h3>
</div>